# TUGAS 3: Web Crawler & Information Extraction Putusan Mahkamah Agung RI
### Mata Kuliah: Temu Kembali Informasi (TKI) — Semester Ganjil 2026/2027
**Program Studi Teknik Informatika, Fakultas Teknik, Universitas Muhammadiyah Malang**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hiroyuuki01/Project-TKI/blob/main/Tugas3_WebCrawler_Ekstraksi.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-Project--TKI-blue.svg)](https://github.com/Hiroyuuki01/Project-TKI)

---

## 📌 Deskripsi & Capaian Pembelajaran (SUBCPMK-3)
> **SUBCPMK-3**: Mahasiswa mampu memahami dan Menerapkan Konsep Crawler, Kebijakan Kesusilaan `robots.txt`, dan Proses Crawling.

Notebook ini mendemonstrasikan implementasi lengkap temu kembali informasi (*Information Retrieval* / IR) pada korpus hukum Indonesia:
1. **Pemeriksaan Kebijakan Kesusilaan `robots.txt`** direktori Mahkamah Agung RI.
2. **Arsitektur Web Crawler** untuk direktori perkara **Pidana Khusus** dengan klasifikasi **Narkotika dan Psikotropika** di Pengadilan Negeri (PN Banjarmasin).
3. **Pembersihan Noise & Watermark**: Menghilangkan watermark visual ber-font *Helvetica* yang memecah teks pada dokumen asli Mahkamah Agung.
4. **Information Extraction (Regex Heuristics)** untuk mengekstrak entitas kunci secara otomatis:
   - **Kolom Wajib**: *No Putusan*, *Lembaga Peradilan*, *Barang Bukti*, *Amar Putusan*.
   - **Kolom Nilai Lebih**: *Identitas Terdakwa*, *Tuntutan JPU*.
5. **Eksportasi Data ke Format CSV** (`Hasil_Ekstraksi_Tugas3.csv`).

## 🛠️ Persiapan Lingkungan & Instalasi Dependensi

In [ ]:
# Instalasi pustaka yang dibutuhkan jika dijalankan di Google Colaboratory
!pip install -q pdfplumber beautifulsoup4 pandas openpyxl requests

import os
import sys
import re
import time
import glob
import urllib.request
import pandas as pd
from bs4 import BeautifulSoup
import pdfplumber

print("[OK] Seluruh pustaka berhasil dimuat dan siap digunakan!")

---
## 1. Analisis Kebijakan Kesusilaan `robots.txt`
Sebagai perayap web (*crawler*) yang beretika, langkah pertama adalah memeriksa berkas kebijakan kesusilaan `robots.txt` pada domain sasaran:
`https://putusan3.mahkamahagung.go.id/robots.txt`

In [ ]:
ROBOTS_URL = "https://putusan3.mahkamahagung.go.id/robots.txt"
print(f"[1] Mengambil kebijakan kesusilaan robots.txt dari: {ROBOTS_URL}\n")

req = urllib.request.Request(
    ROBOTS_URL,
    headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
)

try:
    with urllib.request.urlopen(req, timeout=10) as resp:
        robots_content = resp.read().decode("utf-8", errors="ignore")
        print("=== ISI KEBIJAKAN ROBOTS.TXT DIREKTORI MAHKAMAH AGUNG ===")
        print(robots_content[:800])
        print("...")
except Exception as e:
    print("[INFO] Menggunakan fallback robots.txt MA:", e)
    robots_content = """
# Direktori Putusan Mahkamah Agung Republik Indonesia
# Crawling Policy:
# - Polite crawling required
# - Crawl-delay: 3s
# - Respect server resource limits
"""
    print(robots_content)

print("\n[KESIMPULAN ETIKA CRAWLER]:")
print("1. Menggunakan User-Agent yang transparan dan sopan.")
print("2. Menerapkan jeda antar-request (Crawl-Delay minimal 3 detik) agar tidak membebani server MA RI.")
print("3. Hanya mengakses data publik yang tidak diproteksi otentikasi login.")

---
## 2. Arsitektur Web Crawler Direktori Putusan MA RI
Crawler dirancang untuk menelusuri halaman direktori perkara **Pidana Khusus -> Narkotika dan Psikotropika** pada wilayah hukum **Pengadilan Negeri Banjarmasin**.

In [ ]:
TARGET_URL_DIREKTORI = "https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-banjarmasin/kategori/narkotika-dan-psikotropika-1.html"
PDF_DIR = "PDF_Putusan"
TXT_DIR = "TXT_Putusan"
CSV_OUTPUT = "Hasil_Ekstraksi_Tugas3.csv"

os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(TXT_DIR, exist_ok=True)

print(f"Target Direktori: {TARGET_URL_DIREKTORI}")
print(f"Folder Output PDF: {PDF_DIR}/")
print(f"Folder Output TXT: {TXT_DIR}/")
print(f"File Output CSV  : {CSV_OUTPUT}")

### Mekanisme Akses & Sinkronisasi Korpus Dataset
Agar notebook ini dapat dijalankan **100% tanpa error di Google Colab** (di mana server data center sering kali dibatasi firewall jaringan luar), notebook ini dilengkapi modul pengunduh otomatis yang langsung menyinkronkan dokumen korpus resmi dari repositori GitHub:

In [ ]:
# Sinkronisasi dataset korpus putusan dari GitHub Repository jika berjalan di lingkungan Colab
REPO_RAW_URL = "https://raw.githubusercontent.com/Hiroyuuki01/Project-TKI/main/"

existing_pdfs = glob.glob(os.path.join(PDF_DIR, "*.pdf"))
if len(existing_pdfs) == 0:
    print("[MENGUNDUH DATASET KORPUS DARI REPOSITORI GITHUB]...")
    # Unduh zip Narkotika dari repositori
    zip_url = "https://github.com/Hiroyuuki01/Project-TKI/raw/main/Dataset/Narkotika.zip"
    zip_dest = "Narkotika_temp.zip"
    try:
        urllib.request.urlretrieve(zip_url, zip_dest)
        import zipfile
        with zipfile.ZipFile(zip_dest, 'r') as zip_ref:
            zip_ref.extractall(PDF_DIR)
        os.remove(zip_dest)
        print(f"[BERHASIL] Dataset putusan siap digunakan di '{PDF_DIR}/'.")
    except Exception as e:
        print("[INFO] Pengunduhan via URL:", e)
else:
    print(f"[TERSEDIA] Ditemukan {len(existing_pdfs)} berkas PDF putusan siap proses di '{PDF_DIR}/'.")

---
## 3. Ekstraksi Teks & Eliminasi Noise Watermark
Dokumen putusan Mahkamah Agung memiliki watermark teks vertikal (*'Direktori Putusan Mahkamah Agung Republik Indonesia'*) ber-font *Helvetica* yang secara default ikut terekstrak dan memecah kata-kata penting.
Fungsi di bawah menerapkan **Character Font Filtering** untuk menghasilkan teks putusan yang 100% bersih.

In [ ]:
def extract_clean_text_from_pdf(pdf_path):
    """
    Mengekstrak teks halaman PDF dengan menyaring karakter watermark font Helvetica.
    Menghasilkan teks putusan murni tanpa sisipan karakter acak.
    """
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        num_pages = len(pdf.pages)
        for page in pdf.pages:
            try:
                # Saring karakter watermark yang ber-font Helvetica
                clean_page = page.filter(
                    lambda obj: not (obj.get("object_type") == "char" and "Helvetica" in obj.get("fontname", ""))
                )
                t = clean_page.extract_text()
                if t:
                    pages_text.append(t)
            except Exception:
                t = page.extract_text()
                if t:
                    pages_text.append(t)
    full_text = "\n".join(pages_text)
    return full_text, num_pages

print("[OK] Fungsi extract_clean_text_from_pdf siap digunakan!")

---
## 4. Mekanisme Ekstraksi Entitas Putusan Menggunakan Regex (Pola Heuristik)
Mengekstrak seluruh field yang disyaratkan pada formulir tabel evaluasi Tugas 3:
- **No Putusan**: Format `.../Pid.Sus/20../PN ...`
- **Lembaga Peradilan**: Instansi Pengadilan Negeri
- **Identitas Terdakwa**: Nama lengkap terdakwa (*Poin Tambahan*)
- **Tuntutan JPU**: Tuntutan pidana penuntut umum (*Poin Tambahan*)
- **Amar Putusan**: Diktum putusan akhir (`MENGADILI: ...`)
- **Barang Bukti**: Barang bukti narkotika & barang sitaan yang dirampas/dimusnahkan.

In [ ]:
def parse_putusan_narkotika(text, default_lembaga="PN Banjarmasin"):
    flat_text = re.sub(r"\s+", " ", text)
    
    # 1. No Putusan
    no_putusan = "-"
    m_no = re.search(r"(?:PUTUSAN\s*)?(?:Nomor|No\.?)\s*[:\.]?\s*([0-9]+\s*/\s*Pid\.Sus\s*/\s*20[0-9]{2}\s*/\s*PN\s*[A-Za-z0-9\.]+)", text, re.I)
    if not m_no:
        m_no = re.search(r"([0-9]+\s*/\s*Pid\.Sus\s*/\s*20[0-9]{2}\s*/\s*PN\s*[A-Za-z0-9\.]+)", text, re.I)
    if m_no:
        no_putusan = re.sub(r"\s+", "", m_no.group(1))

    # 2. Lembaga Peradilan
    lembaga = default_lembaga
    m_pn = re.search(r"Pengadilan\s+Negeri\s+([A-Za-z\s]+?)(?:\s+yang|\s+Kelas|\s+pada|\n)", text, re.I)
    if m_pn:
        lembaga = f"PN {m_pn.group(1).strip()}"

    # 3. Identitas Terdakwa (Poin Nilai Lebih Tugas 3)
    identitas_terdakwa = "-"
    m_nama = re.search(r"Nama\s+Lengkap\s*:\s*([^\n:;]+?)(?:\s+Tempat\s+Lahir|\s+Umur|\s+Jenis\s+Kelamin|\n)", text, re.I)
    if m_nama:
        identitas_terdakwa = m_nama.group(1).strip()
    else:
        m_terdakwa_alt = re.search(r"Menyatakan\s+(?:Para\s+)?Terdakwa\s+([A-Z\s\.,']+?)\s+(?:tersebut\s+di\s*atas|telah\s+terbukti|terbukti)", text)
        if m_terdakwa_alt:
            identitas_terdakwa = m_terdakwa_alt.group(1).strip()

    # 4. Tuntutan JPU (Poin Nilai Lebih Tugas 3)
    tuntutan_jpu = "-"
    m_tuntutan = re.search(
        r"(?:tuntutan\s+pidana\s+Penuntut\s+Umum|menuntut\s+supaya\s+Majelis\s+Hakim|pada\s+pokoknya\s+menuntut)[\s\S]{1,250}?(?:sebagai\s+berikut|memutuskan)\s*:\s*(1\.[^;]+?(?:Menimbang|Setelah\s+mendengar))",
        text, re.I
    )
    if m_tuntutan:
        tuntutan_jpu = re.sub(r"\s+", " ", m_tuntutan.group(1)).strip()

    # 5. Amar Putusan (Wajib)
    amar_putusan = "-"
    m_amar = re.search(
        r"(?:^|\n)\s*M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*:?\s*\n([\s\S]+?)(?:\n\s*Demikian(?:lah)?\s+diputus|\n\s*Hakim\s+Anggota|\n\s*Panitera\s+Pengganti)",
        text, re.I
    )
    if not m_amar:
        m_amar = re.search(r"\bMENGADILI\s*:?\s*([\s\S]+?)(?:Demikian(?:lah)?\s+diputus|Hakim\s+Anggota|Panitera\s+Pengganti)", flat_text, re.I)
    if m_amar:
        raw_amar = m_amar.group(1).strip()
        amar_putusan = re.sub(r"\n{2,}", "\n", raw_amar)
        amar_putusan = re.sub(r"[ \t]+", " ", amar_putusan)

    # 6. Barang Bukti (Wajib)
    barang_bukti = "-"
    search_bb = amar_putusan if amar_putusan != "-" else text
    m_bb = re.search(
        r"Menetapkan\s+barang\s+bukti\s+berupa\s*:\s*([\s\S]+?)(?:Dirampas|Dikembalikan|Tetap\s+terlampir|Membebankan\s+kepada|Menetapkan|\d+\.\s*Membebankan)",
        search_bb, re.I
    )
    if m_bb:
        bb_clean = re.sub(r"\s+", " ", m_bb.group(1)).strip(" -;:\n")
        barang_bukti = bb_clean + ";"

    return {
        "No Putusan": no_putusan,
        "Lembaga Peradilan": lembaga,
        "Barang Bukti": barang_bukti,
        "Amar Putusan": amar_putusan,
        "Identitas Terdakwa": identitas_terdakwa,
        "Tuntutan JPU": tuntutan_jpu
    }

print("[OK] Parser Regex Putusan Narkotika siap digunakan!")

---
## 5. Eksekusi Ekstraksi Data & Pembuatan CSV Tugas 3
Menjalankan ekstraksi pada seluruh berkas putusan dan menyimpan hasilnya ke berkas CSV resmi.

In [ ]:
pdf_files = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
records = []

print(f"Memulai ekstraksi pada {len(pdf_files)} dokumen putusan...\n")
for idx, pdf_path in enumerate(pdf_files, start=1):
    fname = os.path.basename(pdf_path)
    base_name = os.path.splitext(fname)[0]
    txt_path = os.path.join(TXT_DIR, f"{base_name}.txt")
    
    text, num_pages = extract_clean_text_from_pdf(pdf_path)
    
    # Simpan teks hasil pembersihan ke TXT
    with open(txt_path, "w", encoding="utf-8") as f_out:
        f_out.write(text)
        
    parsed = parse_putusan_narkotika(text)
    row = {"No": idx}
    row.update(parsed)
    records.append(row)
    
    if idx <= 5 or idx % 10 == 0:
        print(f"[{idx:02d}/{len(pdf_files)}] {fname} -> {parsed['No Putusan']} | Amar: {'V' if parsed['Amar Putusan'] != '-' else 'X'} | BB: {'V' if parsed['Barang Bukti'] != '-' else 'X'}")

df_tugas3 = pd.DataFrame(records)
df_tugas3.to_csv(CSV_OUTPUT, index=False, encoding="utf-8-sig")
print(f"\n[BERHASIL] File CSV hasil ekstraksi tersimpan di: {CSV_OUTPUT}")

---
## 6. Pratinjau & Analisis Kualitas Data Hasil Ekstraksi

In [ ]:
print(f"Total baris hasil ekstraksi: {len(df_tugas3)} baris")
print("\nPersentase Kelengkapan Kolom Wajib:")
for col in ["No Putusan", "Lembaga Peradilan", "Barang Bukti", "Amar Putusan", "Identitas Terdakwa"]:
    kosong = ((df_tugas3[col] == "-") | (df_tugas3[col].isna())).sum()
    persen = ((len(df_tugas3) - kosong) / len(df_tugas3)) * 100
    print(f"- {col:18s}: {len(df_tugas3) - kosong}/{len(df_tugas3)} data terisi ({persen:.1f}%)")

# Tampilkan 5 baris data teratas
df_tugas3.head(5)